CIFAR10 and CIFAR100 dataset
============================

Ref: https://hjweide.github.io/quantifying-uncertainty-in-neural-networks

1. Red objects: cars or apples?
2. Green objects: frogs? or apples?

![](../diagrams/weather.png)

Reference slides
----------------

First 7 slides from above link

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
# Retina display
%config InlineBackend.figure_format = 'retina'


In [ ]:
from tueplots import bundles
plt.rcParams.update(bundles.beamer_moml())


# Also add despine to the bundle using rcParams
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False

# Increase font size to match Beamer template
plt.rcParams['font.size'] = 16
# Make background transparent
plt.rcParams['figure.facecolor'] = 'none'

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression

X, y = make_moons(n_samples=1000, noise=0.1, random_state=0)

clf = LogisticRegression()

clf.fit(X, y)

In [ ]:
p_hats = clf.predict_proba(X)[:, 1]
p_hats[:10]

In [ ]:
true_labels = y
true_labels[:10]

In [ ]:
df = pd.DataFrame({'p_hat': p_hats, 'y': true_labels})
df.head()

In [ ]:
# Let us look at samples where p_hat is close to 0.5 (say between 0.45 and 0.55)
df_close_phat_val = df[(df.p_hat > 0.45) & (df.p_hat < 0.55)]
df_close_phat_val.head()

In [ ]:
len(df_close_phat_val)

In [ ]:
# Histogram of p_hat values
df.p_hat.hist(bins=50)
plt.xlabel('$\hat{p}$')
plt.ylabel('Frequency')
plt.title('Histogram of $\hat{p}$ values');

In [ ]:
# Out of these len(df_close_phat_val) samples, how many are actually positive? 

df_close_phat_val.y.sum()

In [ ]:
# Fraction of positive samples
df_close_phat_val.y.sum() / len(df_close_phat_val)

In [ ]:
# This is the reliability of the classifier at p_hat = 0.5

In [ ]:
bins = np.linspace(0, 1, 11)
bins

In [ ]:
# Let us add a column to the dataframe that tells us which bin each sample belongs to

df['bin'] = pd.cut(df.p_hat, bins=bins)
df.head()

In [ ]:
# Let us compute the reliability of the classifier in each bin

df.groupby('bin').y.mean()

In [ ]:
# Calculate the reliability diagram points
mean_predicted_probs = df.groupby('bin').p_hat.mean()
fraction_of_positives = df.groupby('bin').y.mean()


In [ ]:
bin_sizes = df.groupby('bin').size()
bin_sizes

In [ ]:
ece = (np.abs(mean_predicted_probs - fraction_of_positives) * bin_sizes / len(df)).sum()
ece

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k:')
plt.plot(mean_predicted_probs, fraction_of_positives, 'o-')
plt.xlabel('Predicted probability')
plt.ylabel('Fraction of positives')
plt.title(f'Reliability Diagram (ECE={ece:.4f})')
plt.gca().set_aspect('equal')

# Plot the bar also
plt.bar(mean_predicted_probs, fraction_of_positives, width=0.1, alpha=0.1)

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(df['y'], df['p_hat'], n_bins=10, strategy='uniform')


In [ ]:
assert np.allclose(prob_true, fraction_of_positives)

In [ ]:
assert np.allclose(prob_pred, mean_predicted_probs)

In [ ]:
pd.DataFrame({'prob_true': prob_true, 'prob_pred': prob_pred})


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

def plot_reliability_diagram(classifier, X, y, n_bins=10, ax=None):
    """
    Generate a reliability diagram and calculate ECE for a classifier.

    Parameters:
    - classifier: A scikit-learn classifier with a `predict_proba` method.
    - X: Input features for the dataset.
    - y: True labels for the dataset.
    - n_bins: Number of bins for the reliability diagram.
    - ax: Matplotlib axes object to plot on.

    Returns:
    - ece: Expected Calibration Error (ECE).
    """
    if ax is None:
        fig, ax = plt.subplots()

    # Fit the classifier
    classifier.fit(X, y)

    # Predict probabilities
    p_hats = classifier.predict_proba(X)[:, 1]

    # Create a DataFrame
    df = pd.DataFrame({'p_hat': p_hats, 'y': y})

    # Create bins based on predicted probabilities
    bins = np.linspace(0, 1, n_bins + 1)
    df['bin'] = pd.cut(df['p_hat'], bins=bins)
    
    mean_predicted_probs = df.groupby('bin').p_hat.mean()
    fraction_of_positives = df.groupby('bin').y.mean()
    
    # Fill NANs with 0
    mean_predicted_probs = mean_predicted_probs.fillna(0)
    fraction_of_positives = fraction_of_positives.fillna(0)

    # Calculate the Expected Calibration Error (ECE) weighted by bin sizes
    bin_sizes = df.groupby('bin').size()
    
    # Ensure bin_sizes align with prob_true and prob_pred
    #bin_sizes = bin_sizes.reindex(range(n_bins), fill_value=0)
    #return bin_sizes, mean_predicted_probs, fraction_of_positives

    ece = (np.abs(mean_predicted_probs - fraction_of_positives) * bin_sizes / bin_sizes.sum()).sum()

    # Create the reliability diagram plot
    #plt.figure(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], 'k:')
    ax.scatter(mean_predicted_probs, fraction_of_positives)
    # Plot bar also
    ax.bar(mean_predicted_probs, fraction_of_positives, width=0.1, alpha=0.1)
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'Calibration Curve (ECE={ece:.4f})')
    #plt.gca().set_aspect('equal')
    return ece


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibrationDisplay

# Generate the dataset
X, y = make_moons(n_samples=1000, noise=0.1, random_state=0)

# List of classifiers to evaluate
classifiers = [
    ('Logistic Regression', LogisticRegression()),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=0)),
    ('Support Vector Classifier', SVC(probability=True)),
    ('K-Nearest Neighbors', KNeighborsClassifier(n_neighbors=3))
]

fig, axarr = plt.subplots(figsize=(6, 6), nrows=len(classifiers), ncols=2)

# Loop through the classifiers and generate reliability diagrams
for i, (name, classifier) in enumerate(classifiers):
    ece = plot_reliability_diagram(classifier, X, y, ax=axarr[i, 0])
    print(f"Classifier: {name}")
    print(f"ECE: {ece:.4f}")


    
    CalibrationDisplay.from_estimator(
        classifier,
        X,
        y,
        n_bins=10,
        ax = axarr[i, 1],
        name=name
    )
    print("*"*20)
    print("*"*20)
    print("\n" + "=" * 50 + "\n")

### Multiclass reliability diagram

Slide 26 from https://classifier-calibration.github.io/assets/slides/clacal_tutorial_ecmlpkdd_2020_intro.pdf

In [ ]:
# Now, multi-class classification and reliability diagrams

from sklearn.datasets import make_classification, make_blobs

X, y = make_blobs(n_samples=1000, centers=3, n_features=2, random_state=0)

# Plot the dataset
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis')

In [ ]:
clf = LogisticRegression()
clf.fit(X, y)

In [ ]:
preds = pd.DataFrame(clf.predict_proba(X))

In [ ]:
preds

In [ ]:
preds['y'] = y

In [ ]:
preds.head(10)

In [ ]:
# Syntax highlighting to show highest amongst the three columns
most_probable_class = preds[[0, 1, 2]].idxmax(axis=1)

# Add the most probable class to the DataFrame
preds['y_hat'] = most_probable_class


In [ ]:
preds.head()

In [ ]:
# Let us simplify dataframe to only include the most probable class probability and whether the prediction was correct or not
new_df = preds.copy()
new_df["Correct"] = new_df.y == new_df.y_hat
new_df["Most_Probable_Class_Probability"] = new_df[[0, 1, 2]].max(axis=1)

new_df.head()

In [ ]:
# drop the other columns
new_df = new_df.drop(columns=[0, 1, 2, 'y', 'y_hat'])
new_df.head()

In [ ]:
# Let us look at samples where the most probable class probability is close to 0.5 (say between 0.45 and 0.55)
df_close_phat_val = new_df[(new_df.Most_Probable_Class_Probability > 0.45) & (new_df.Most_Probable_Class_Probability < 0.55)]
df_close_phat_val.head()

In [ ]:
df_close_phat_val.Correct.sum() / len(df_close_phat_val)

In [ ]:
# Now, let us compute the reliability of the classifier in each bin
bins = np.linspace(0, 1, 11)

# Cuts the data by the bins
new_df['bin'] = pd.cut(new_df.Most_Probable_Class_Probability, bins=bins)


In [ ]:
new_df.head(10)

In [ ]:
# Calculate the reliability diagram points
mean_predicted_probs = new_df.groupby('bin').Most_Probable_Class_Probability.mean()
fraction_of_positives = new_df.groupby('bin').Correct.mean()

bin_sizes = new_df.groupby('bin').size()

ece = (np.abs(mean_predicted_probs - fraction_of_positives) * bin_sizes / len(new_df)).sum()


In [ ]:
ece

In [ ]:
# Plot the reliability diagram
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k:')
plt.plot(mean_predicted_probs, fraction_of_positives, 'o-')
plt.xlabel('Predicted probability')
plt.ylabel('Fraction of positives')
plt.title(f'Reliability Diagram (ECE={ece:.4f})')

In [ ]:
# Histogram of p_hat values

new_df.Most_Probable_Class_Probability.hist(bins=50)
plt.xlim(0, 1)

### Reliability Diagrams for regression


In [ ]:
# Now let us look at the above for regression with heteroscedastic aleatoric uncertainty

X = np.linspace(-3, 3, 1000).reshape(-1, 1)
y = np.sin(X.flatten()) + np.random.normal(0, 0.1, X.shape[0])

plt.scatter(X, y, s=10, alpha=0.5)

In [ ]:
# Use BayesianRidge for regression
from sklearn.linear_model import BayesianRidge

reg = BayesianRidge()
reg.fit(X, y)

In [ ]:
# Predict on linspace
X_test = np.linspace(-3, 3, 100).reshape(-1, 1)
y_pred, y_std = reg.predict(X_test, return_std=True)

# Plot the predictions
plt.plot(X_test, y_pred)
plt.fill_between(X_test.flatten(), y_pred - y_std, y_pred + y_std, alpha=0.5)
plt.scatter(X, y, alpha=0.1)

In [ ]:
# Within 1\sigma we would like to see 68% of the data points
# Within 2\sigma we would like to see 95% of the data points

# Let us see how many points are within k*sigma of the mean
k = 1
y_pred, y_std = reg.predict(X, return_std=True)
# Check how many points are within k*sigma of the mean
within_k_sigma = np.abs(y - y_pred) < k * y_std
# Fraction of points within k*sigma of the mean
within_k_sigma.mean()

In [ ]:
def within_k_sigma(y, y_pred, y_std, k=1):
    """
    Return the fraction of points within k*sigma of the mean.
    """
    return np.abs(y - y_pred) < k * y_std

In [ ]:
# Plot within_k_sigma for k=1, 2, 3, ..

ks = np.arange(1, 6)
within_k_sigma_fractions = [within_k_sigma(y, y_pred, y_std, k=k).mean() for k in ks]

plt.plot(ks, within_k_sigma_fractions, 'o-')
plt.xlabel('k')
plt.ylabel('Fraction of points within k*sigma of the mean')

In [ ]:
# but, ideally we want 95% of the points to be within 2*sigma of the mean or within 95% CI
# we need a function to map CI to k

from scipy.stats import norm

def ci_to_k(ci):
    """
    Return the value of k corresponding to the given confidence interval.
    """
    return norm.ppf((1 + ci) / 2)

In [ ]:
ci_to_k(0.95)

In [ ]:
ci_to_k(0.68)

In [ ]:
# make reliability diagram for regression

percentiles = np.linspace(0, 100, 11)
quantiles = percentiles / 100

ks = ci_to_k(quantiles)

# Get fraction of points within k*sigma of the mean for each k
within_k_sigma_fractions = [within_k_sigma(y, y_pred, y_std, k=k).mean() for k in ks]

plt.plot(quantiles, within_k_sigma_fractions, 'o-')
# Plot ideal
plt.plot([0, 1], [0, 1], 'k:')